# Model Training

In [2]:
# Data Ingestion step
import pandas as pd
df = pd.read_csv(r'data/gemstone.csv')
data = df.rename(columns={'Unnamed: 0' : "id"})
data.head()

,id,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [3]:
data.drop(columns=['id'], inplace=True)

In [4]:
data.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [5]:
data['color'].value_counts()

color
G    11292
E     9797
F     9542
H     8304
D     6775
I     5422
J     2808
Name: count, dtype: int64

In [6]:
# Independent and dependent features
X = data.drop(labels=['price'], axis=1)
Y = data[['price']]

In [7]:
# Define which columns should be original = endcoded and which should be scaled.
categorical_cols = X.select_dtypes(include='object').columns
numerical_cols = X.select_dtypes(exclude='object').columns

In [8]:
# Define the custom ranking for each ordinal variable
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_categories = ['D', 'E', 'F', 'G', 'H', 'I', 'J']
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

In [9]:
from sklearn.impute import SimpleImputer ## Handling missing values
from sklearn.preprocessing import StandardScaler ## Handling Feature scaling
from sklearn.preprocessing import OrdinalEncoder ## Ordinal Encoding
# pipelines
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [10]:
# Numerical Pipeline
num_pipeline = Pipeline(
             steps=[
              ('imputer', SimpleImputer(strategy='median')),
               ('scaler', StandardScaler())   
          
             ]    
) 

# Categorical Pipeline

cat_pipeline = Pipeline(
              steps=[
               ('imputer', SimpleImputer(strategy='most_frequent')),
               ('ordinal_encoder', OrdinalEncoder(categories=[
                 cut_categories,color_categories, clarity_categories])),
               ('scaler', StandardScaler())
              ]
     
)

preprocessor = ColumnTransformer([
('num_pipeline', num_pipeline, numerical_cols),
('cat_pipline', cat_pipeline, categorical_cols)
])


In [11]:
# Train Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.30, random_state=42)


In [12]:
X_train =pd.DataFrame(preprocessor.fit_transform(X_train), columns=preprocessor.get_feature_names_out())
X_test =pd.DataFrame(preprocessor.transform(X_test), columns=preprocessor.get_feature_names_out())

In [13]:
X_train.head()

,num_pipeline__carat,num_pipeline__depth,num_pipeline__table,num_pipeline__x,num_pipeline__y,num_pipeline__z,cat_pipline__cut,cat_pipline__color,cat_pipline__clarity
0,0.862659,-0.311437,-0.207099,1.055581,0.986556,0.968253,0.982948,0.825977,1.184867
1,-1.029889,0.178549,-0.656213,-1.207734,-1.202544,-1.168276,0.982948,-0.938373,-0.030994
2,0.862659,0.458541,-0.207099,0.904099,0.951670,0.982309,0.982948,-0.938373,0.576936
3,0.021527,0.598537,-1.105327,0.164512,0.192898,0.251391,0.982948,-0.350256,-1.246854
4,-0.020530,-0.031445,-0.656213,0.182333,0.184176,0.181110,0.982948,1.414093,1.184867


In [14]:
X_test.head()

,num_pipeline__carat,num_pipeline__depth,num_pipeline__table,num_pipeline__x,num_pipeline__y,num_pipeline__z,cat_pipline__cut,cat_pipline__color,cat_pipline__clarity
0,-1.177087,0.248547,-0.656213,-1.573073,-1.516519,-1.505623,0.982948,0.237860,1.792797
1,-0.462124,-1.221412,-0.207099,-0.263201,-0.278064,-0.395190,-0.810396,-0.350256,1.184867
2,-0.840634,0.248547,-1.105327,-0.869128,-0.871127,-0.830930,0.982948,-0.938373,1.184867
3,-0.777549,-0.661428,-0.207099,-0.726557,-0.740304,-0.788761,0.086276,-0.938373,1.184867
4,1.577621,0.388543,-1.105327,1.518937,1.422631,1.502385,0.982948,-0.938373,-1.246854


In [15]:
# Model Training
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [16]:
regression = LinearRegression()
regression.fit(X_train, y_train)

LinearRegression()

In [17]:
regression.coef_

array([[5087.10788988, -112.35468241,  -61.24955364, -945.40540287,
          29.0428686 ,  -16.84668863,  134.40954193, -551.1286418 ,
         829.19699601]])

In [18]:
regression.intercept_

array([3951.49531225])

In [19]:
import numpy as np
def evaluate_model(true, predicted):
     mae = mean_absolute_error(true, predicted)
     mse = mean_squared_error(true, predicted)
     rmse = np.sqrt(mean_squared_error(true, predicted))
     r2_square = r2_score(true, predicted)
     return mae, mse, rmse, r2_square

In [20]:
# Train multiple models
models = {
     'LinearRegression': LinearRegression(),
     'Lasso' : Lasso(),
     'Ridge' : Ridge(),
     'ElasticNet' : ElasticNet()
}

model_list = []
r2_list = []
for i in range(len(list(models))):
     model = list(models.values())[i]
     model.fit(X_train, y_train)
     
     # Make Prediction
     y_pred = model.predict(X_test)
     mae, mse, rmse, r2_square = evaluate_model(y_test, y_pred)
     print(list(models.keys())[i])
     model_list.append(list(models.keys())[i])
     print("Model Training Performance")
     print("RMSE:", rmse)
     print("MAE:", mae)
     print("R2 score:", r2_square)
     print("MSE:", mse)

LinearRegression
Model Training Performance
RMSE: 1201.2077317517571
MAE: 802.2014845937462
R2 score: 0.9074824695469136
MSE: 1442900.0148202016
Lasso
Model Training Performance
RMSE: 1201.4563058256667
MAE: 803.4602652081206
R2 score: 0.9074441750232354
MSE: 1443497.2548082578
Ridge
Model Training Performance
RMSE: 1201.2184531323269
MAE: 802.3093890456123
R2 score: 0.9074808180089593
MSE: 1442925.7721456203
ElasticNet
Model Training Performance
RMSE: 1591.2038201368925
MAE: 1064.8451403518977
R2 score: 0.8376548123849741
MSE: 2531929.59721824


In [21]:
import numpy as np
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Function to evaluate model performance
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)  # Fix: Directly use mse instead of recomputing
    r2_square = r2_score(true, predicted)
    return mae, mse, rmse, r2_square  # Fix: Returning all values correctly

# Dictionary of models
models = {
    'LinearRegression': LinearRegression(),
    'Lasso': Lasso(),
    'Ridge': Ridge(),
    'ElasticNet': ElasticNet()
}

model_list = []
r2_list = []

# Train and evaluate each model
for name, model in models.items():
    model.fit(X_train, y_train)  # Train the model

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate model
    mae, mse, rmse, r2_square = evaluate_model(y_test, y_pred)  # Fix: Correct unpacking

    # Store model name and R² score
    model_list.append(name)
    r2_list.append(r2_square)

    # Display results
    print(f"Model: {name}")
    print("Model Training Performance:")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"R2 Score: {r2_square:.4f}")
    print("-" * 40)


Model: LinearRegression
Model Training Performance:
RMSE: 1201.2077
MAE: 802.2015
MSE: 1442900.0148
R2 Score: 0.9075
----------------------------------------
Model: Lasso
Model Training Performance:
RMSE: 1201.4563
MAE: 803.4603
MSE: 1443497.2548
R2 Score: 0.9074
----------------------------------------
Model: Ridge
Model Training Performance:
RMSE: 1201.2185
MAE: 802.3094
MSE: 1442925.7721
R2 Score: 0.9075
----------------------------------------
Model: ElasticNet
Model Training Performance:
RMSE: 1591.2038
MAE: 1064.8451
MSE: 2531929.5972
R2 Score: 0.8377
----------------------------------------


In [22]:
model_list

['LinearRegression', 'Lasso', 'Ridge', 'ElasticNet']